# Alberi di classificazione e random forest

*Metodi Matematici e Statistici per le Decisioni Economiche · A.A. 2026/27 · Lezione 4*

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincnardelli/msd/blob/main/codice/4/alberi_classificazione.ipynb)

Il codice della lezione: regressione logistica, albero e random forest a confronto su `default.csv` e `college.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

DATI = "https://www.vincnardelli.com/msd/dati/"

In [ ]:
def metriche(effettivo, predetto):
    """Confusion matrix (righe = predetto, colonne = effettivo), accuracy, sensitività, specificità."""
    cm = pd.crosstab(predetto, effettivo, rownames=["Predetto"], colnames=["Effettivo"])
    tp = ((predetto == 1) & (effettivo == 1)).sum()
    tn = ((predetto == 0) & (effettivo == 0)).sum()
    fp = ((predetto == 1) & (effettivo == 0)).sum()
    fn = ((predetto == 0) & (effettivo == 1)).sum()
    print(cm, "\n")
    print(f"Accuracy:    {(tp + tn) / len(effettivo):.4f}")
    print(f"Sensitività: {tp / (tp + fn):.4f}")
    print(f"Specificità: {tn / (tn + fp):.4f}")
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

## Default: logistica contro albero

In [ ]:
Default = pd.read_csv(DATI + "default.csv")
Default["y"] = (Default["default"] == "Yes").astype(int)

glm_model = smf.logit("y ~ balance + income", data=Default).fit()
print(glm_model.summary())
metriche(Default["y"], (glm_model.predict(Default) > 0.5).astype(int));

In [ ]:
X = Default[["balance", "income"]]
tree_model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=1).fit(X, Default["y"])
metriche(Default["y"], tree_model.predict(X))

plt.figure(figsize=(16, 7))
plot_tree(tree_model, feature_names=X.columns, class_names=["No", "Yes"], filled=True, rounded=True, fontsize=8)
plt.show()

## College: logistica, albero e random forest

In [ ]:
College = pd.read_csv(DATI + "college.csv")
College["priv"] = (College["Private"] == "Yes").astype(int)
X = College[["Room.Board", "Outstate", "PhD"]]

glm_model = smf.logit("priv ~ Q('Room.Board') + Outstate + PhD", data=College).fit()
print(glm_model.summary())
metriche(College["priv"], (glm_model.predict(College) > 0.5).astype(int));

In [ ]:
tree_model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=1).fit(X, College["priv"])
metriche(College["priv"], tree_model.predict(X))

plt.figure(figsize=(16, 7))
plot_tree(tree_model, feature_names=X.columns, class_names=["No", "Yes"], filled=True, rounded=True, fontsize=8)
plt.show()

In [ ]:
rf_model = RandomForestClassifier(n_estimators=500, oob_score=True, random_state=123).fit(X, College["priv"])
rf_pred = rf_model.oob_decision_function_[:, 1] > 0.5     # previsioni out-of-bag
metriche(College["priv"], rf_pred.astype(int))

pd.Series(rf_model.feature_importances_, index=X.columns).sort_values().plot.barh()
plt.xlabel("importanza (riduzione media dell'indice di Gini)")
plt.show()